# 🚀 02 - Fine-Tuning del Modelo de Lenguaje (RotBot English Coach)

Este notebook guía el proceso de lanzamiento, monitoreo y registro del trabajo de **Supervised Fine-Tuning (SFT)** para **RotBot**.

### Opciones de Entrenamiento:
1. **Google AI Studio (Gemini 1.5 Flash):** *(Recomendado)* Alta velocidad, costos mínimos y excelente compresión multilingüe.
2. **OpenAI API (GPT-4o-mini):** Alternativa para fine-tuning estándar con formato ChatML.

## 1. Configuración de Dependencias y Variables de Entorno

In [ ]:
import os
import sys
import time
import json
from pathlib import Path
from dotenv import load_dotenv, set_key

PROJECT_ROOT = Path(os.path.abspath("")).resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

ENV_PATH = os.path.join(PROJECT_ROOT, ".env")
load_dotenv(ENV_PATH)

TRAIN_JSONL_PATH = os.path.join(PROJECT_ROOT, "data", "processed", "train.jsonl")
VAL_JSONL_PATH = os.path.join(PROJECT_ROOT, "data", "processed", "val.jsonl")

print(f"📁 Raíz del Proyecto: {PROJECT_ROOT}")
print(f"📄 Train JSONL: {TRAIN_JSONL_PATH} (Existe: {os.path.exists(TRAIN_JSONL_PATH)})")
print(f"📄 Val JSONL:   {VAL_JSONL_PATH} (Existe: {os.path.exists(VAL_JSONL_PATH)})")

## 2. Pre-Flight Checks & Estimación de Pasos de Entrenamiento
Calculamos las métricas de pasos de entrenamiento por época para asegurar convergencia óptima.

In [ ]:
with open(TRAIN_JSONL_PATH, 'r', encoding='utf-8') as f:
    train_samples = [json.loads(l) for l in f if l.strip()]
with open(VAL_JSONL_PATH, 'r', encoding='utf-8') as f:
    val_samples = [json.loads(l) for l in f if l.strip()]

N_TRAIN = len(train_samples)
N_VAL = len(val_samples)
DEFAULT_BATCH_SIZE = 4
DEFAULT_EPOCHS = 5
STEPS_PER_EPOCH = N_TRAIN // DEFAULT_BATCH_SIZE
TOTAL_STEPS = STEPS_PER_EPOCH * DEFAULT_EPOCHS

print("=== 📋 RESUMEN PRE-ENTRENAMIENTO ===")
print(f"• Ejemplos de Entrenamiento: {N_TRAIN}")
print(f"• Ejemplos de Validación:   {N_VAL}")
print(f"• Batch Size sugerido:      {DEFAULT_BATCH_SIZE}")
print(f"• Épocas sugeridas:         {DEFAULT_EPOCHS}")
print(f"• Pasos por época:          {STEPS_PER_EPOCH}")
print(f"• Total de Pasos (Steps):   {TOTAL_STEPS}")

## 3. Opción A: Fine-Tuning con Google Gemini (AI Studio)
Ajustamos el modelo base `models/gemini-1.5-flash-001-tuning` utilizando el SDK de Google Generative AI.

In [ ]:
import google.generativeai as genai

gemini_api_key = os.getenv("GEMINI_API_KEY")
if not gemini_api_key or gemini_api_key.startswith("your_"):
    print("⚠️ Por favor, define GEMINI_API_KEY en tu archivo .env para entrenar con Google AI Studio.")
else:
    genai.configure(api_key=gemini_api_key)
    print("✅ Gemini API configurada con éxito.")
    
    # Preparación de pares para Gemini Supervised Tuning
    gemini_training_pairs = []
    for record in train_samples:
        if "messages" in record:
            u_text = next((m["content"] for m in record["messages"] if m["role"] == "user"), "")
            a_text = next((m["content"] for m in record["messages"] if m["role"] == "assistant"), "")
            if u_text and a_text:
                gemini_training_pairs.append({"text_input": u_text, "output": a_text})
                
    print(f"✅ {len(gemini_training_pairs)} ejemplos preparados para Gemini Tuning.")

In [ ]:
# Lanzamiento y Monitoreo del Job en Google AI Studio
# (Descomenta las siguientes líneas cuando estés listo para ejecutar el entrenamiento)

"""
tuned_model_name = "rotbot-english-coach-v1"

print(f"🚀 Iniciando Fine-Tuning Job: {tuned_model_name}...")
operation = genai.create_tuned_model(
    source_model="models/gemini-1.5-flash-001-tuning",
    training_data=gemini_training_pairs,
    id=tuned_model_name,
    display_name="RotBot English Coach v1",
    description="Sarcastic and clever English coach model for Spanish speakers",
    epoch_count=DEFAULT_EPOCHS,
    batch_size=DEFAULT_BATCH_SIZE,
    learning_rate=0.001,
)

print(f"Operación ID: {operation.name}")
print("Monitoreando progreso (esto puede tardar unos minutos)...")

for status in operation.wait_bar():
    time.sleep(10)

result_model = operation.result()
model_id = result_model.name
print(f"🎉 ¡Entrenamiento completado! Model ID: {model_id}")

# Guardar en .env automáticamente
set_key(ENV_PATH, "TUNED_MODEL_ID", model_id)
print(f"💾 TUNED_MODEL_ID guardado automáticamente en .env")
"""

## 4. Opción B: Fine-Tuning con OpenAI (GPT-4o-mini)
Subida directa de archivos `train.jsonl` y `val.jsonl` a OpenAI Files API y lanzamiento del SFT Job.

In [ ]:
from openai import OpenAI

openai_api_key = os.getenv("OPENAI_API_KEY")
if not openai_api_key or openai_api_key.startswith("your_"):
    print("ℹ️ OPENAI_API_KEY no configurada. Si deseas usar OpenAI, colócala en tu .env.")
else:
    client = OpenAI(api_key=openai_api_key)
    print("✅ Cliente OpenAI inicializado.")
    
    # Subida de datasets
    print("Subiendo train.jsonl...")
    with open(TRAIN_JSONL_PATH, "rb") as f:
        train_file = client.files.create(file=f, purpose="fine-tune")
    print(f"✅ Training File ID: {train_file.id}")
    
    print("Subiendo val.jsonl...")
    with open(VAL_JSONL_PATH, "rb") as f:
        val_file = client.files.create(file=f, purpose="fine-tune")
    print(f"✅ Validation File ID: {val_file.id}")

In [ ]:
# Lanzamiento del Job en OpenAI (Descomenta para ejecutar)
"""
job = client.fine_tuning.jobs.create(
    training_file=train_file.id,
    validation_file=val_file.id,
    model="gpt-4o-mini-2024-07-18",
    suffix="rotbot-coach-v1",
    hyperparameters={
        "n_epochs": DEFAULT_EPOCHS
    }
)
print(f"🚀 Fine-Tuning Job creado con ID: {job.id}")
print(f"Estado inicial: {job.status}")

# Esperar hasta que finalice y registrar ID
while True:
    status_job = client.fine_tuning.jobs.retrieve(job.id)
    print(f"Status: {status_job.status}...")
    if status_job.status in ('succeeded', 'failed', 'cancelled'):
        if status_job.status == 'succeeded':
            final_model = status_job.fine_tuned_model
            print(f"🎉 Modelo entrenado: {final_model}")
            set_key(ENV_PATH, "TUNED_MODEL_ID", final_model)
            print(f"💾 TUNED_MODEL_ID guardado en .env")
        break
    time.sleep(30)
"""

## 5. Próximo Paso: Evaluación de Rendimiento
Una vez completado el entrenamiento y registrado el `TUNED_MODEL_ID`, continúa con el notebook **`03_chat_evaluation.ipynb`** para correr la batería de pruebas pedagógicas y conversar con RotBot.